# Detection Models

In [27]:
from ultralytics import YOLO
detect_face_plate_sign_model = YOLO("https://huggingface.co/Panoramax/detect_face_plate_sign/resolve/main/yolov8s_panoramax.pt")
detect_face_plate_sign_model.to('mps')

Found https://huggingface.co/Panoramax/detect_face_plate_sign/resolve/main/yolov8s_panoramax.pt locally at /Users/lukasappel/Documents/HTW-Studium/Applied Artificial Intelligence/Projekt/AAIProject/weights/yolov8s_panoramax.pt


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_s

In [29]:
detect_face_plate_sign_model(source='./streetview_images/6058367b-b429-4f34-8955-674ace44c18b.jpg',save=True)


image 1/1 /Users/lukasappel/Documents/HTW-Studium/Applied Artificial Intelligence/Projekt/AAIProject/streetview_images/streetview_images/6058367b-b429-4f34-8955-674ace44c18b.jpg: 1152x2048 1 0, 299.8ms
Speed: 146.4ms preprocess, 299.8ms inference, 32.6ms postprocess per image at shape (1, 3, 1152, 2048)
Results saved to /Users/lukasappel/Documents/HTW-Studium/Applied Artificial Intelligence/Projekt/AAIProject/runs/detect/predict2


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: '0', 1: '1', 2: '2'}
 obb: None
 orig_img: array([[[ 47,  29,  18],
         [ 47,  29,  18],
         [ 47,  29,  18],
         ...,
         [218, 205, 203],
         [224, 218, 219],
         [220, 219, 221]],
 
        [[ 47,  29,  18],
         [ 47,  29,  18],
         [ 47,  29,  18],
         ...,
         [214, 199, 196],
         [219, 213, 214],
         [220, 218, 218]],
 
        [[ 47,  29,  18],
         [ 47,  29,  18],
         [ 47,  29,  18],
         ...,
         [210, 191, 186],
         [217, 207, 207],
         [219, 215, 214]],
 
        ...,
 
        [[117, 119, 119],
         [128, 130, 130],
         [134, 136, 136],
         ...,
         [119, 129, 139],
         [116, 126, 136],
         [111, 121, 131]],
 
        [[118, 120, 120],
         [126, 128, 128],
         [124, 126, 126],
         ...,
       

# Classification Model

In [1]:
from ultralytics import YOLO

classification_model = YOLO("Panoramax/classify_fr_road_signs")
classification_model.to('mps')

FileNotFoundError: 'Panoramax/classify_fr_road_signs' does not exist

In [2]:
import cv2
from ultralytics import YOLO

# Load your two models
detector = YOLO("yolov8s_panoramax.pt")    # Finds the signs
classifier = YOLO("https://huggingface.co/Panoramax/classify_fr_road_signs/resolve/main/best.pt")  # Classifies the specific sign

def detect_and_classify(image_path):
    # 1. Load the original image
    img = cv2.imread(image_path)
    
    # 2. Run detection
    det_results = detector(img)[0]  # Get first result
    
    final_results = []

    # 3. Iterate over detected bounding boxes
    for box in det_results.boxes:
        # Get coordinates (convert to integers)
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = box.conf[0].item()
        
        # 4. Crop the region of interest (ROI)
        # Note: handle edge cases where crop might be empty
        crop = img[y1:y2, x1:x2]
        
        if crop.size > 0:
            # 5. Run classification on the crop
            cls_results = classifier(crop)[0]
            
            # Get the top class name and confidence
            top_class_idx = cls_results.probs.top1
            class_name = cls_results.names[top_class_idx]
            cls_conf = cls_results.probs.top1conf.item()

            final_results.append({
                "box": [x1, y1, x2, y2],
                "det_conf": conf,
                "label": class_name,
                "cls_conf": cls_conf
            })

    return final_results

# Usage
output = detect_and_classify("./test1.png")
for res in output:
    print(f"Found {res['label']} at {res['box']} (Conf: {res['cls_conf']:.2f})")

Found https://huggingface.co/Panoramax/classify_fr_road_signs/resolve/main/best.pt locally at /Users/lukasappel/Documents/HTW-Studium/Applied Artificial Intelligence/Projekt/AAIProject/weights/best.pt

0: 2048x2048 3 0s, 1160.7ms
Speed: 21.5ms preprocess, 1160.7ms inference, 9.9ms postprocess per image at shape (1, 3, 2048, 2048)

0: 224x224 B1 1.00, B1j 0.00, Bzz 0.00, AB3 0.00, z0 0.00, 38.4ms
Speed: 9.3ms preprocess, 38.4ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 B1 1.00, B1j 0.00, AB3 0.00, PEI 0.00, Bzz 0.00, 31.8ms
Speed: 0.9ms preprocess, 31.8ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 B1 1.00, B1j 0.00, Bzz 0.00, AB3 0.00, B6a1 0.00, 32.5ms
Speed: 0.7ms preprocess, 32.5ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)
Found B1 at [110, 333, 123, 346] (Conf: 1.00)
Found B1 at [12, 318, 31, 337] (Conf: 1.00)
Found B1 at [348, 321, 359, 333] (Conf: 1.00)


In [25]:
detector(source='./streetview_images/6058367b-b429-4f34-8955-674ace44c18b.jpg',save=True)[0]


image 1/1 /Users/lukasappel/Documents/HTW-Studium/Applied Artificial Intelligence/Projekt/AAIProject/streetview_images/streetview_images/6058367b-b429-4f34-8955-674ace44c18b.jpg: 192x320 (no detections), 84.4ms
Speed: 0.6ms preprocess, 84.4ms inference, 0.1ms postprocess per image at shape (1, 3, 192, 320)
Results saved to /Users/lukasappel/Documents/HTW-Studium/Applied Artificial Intelligence/Projekt/AAIProject/runs/detect/predict


ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'panneau', 1: 'panonceau'}
obb: None
orig_img: array([[[ 47,  29,  18],
        [ 47,  29,  18],
        [ 47,  29,  18],
        ...,
        [218, 205, 203],
        [224, 218, 219],
        [220, 219, 221]],

       [[ 47,  29,  18],
        [ 47,  29,  18],
        [ 47,  29,  18],
        ...,
        [214, 199, 196],
        [219, 213, 214],
        [220, 218, 218]],

       [[ 47,  29,  18],
        [ 47,  29,  18],
        [ 47,  29,  18],
        ...,
        [210, 191, 186],
        [217, 207, 207],
        [219, 215, 214]],

       ...,

       [[117, 119, 119],
        [128, 130, 130],
        [134, 136, 136],
        ...,
        [119, 129, 139],
        [116, 126, 136],
        [111, 121, 131]],

       [[118, 120, 120],
        [126, 128, 128],
        [124, 126, 126],
        ...,
        [124, 134, 144],
        [120, 130, 14